# Movie Recommendation System

# Model Building

## Objectives

- Convert movie tags into vectors
- Compute similarity between movies
- Build recommendation function

In [1]:
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import pickle

In [2]:
import ast
import warnings

warnings.filterwarnings("ignore")

In [3]:
movies = pd.read_csv("../data/tmdb_5000_movies.csv")
credits = pd.read_csv("../data/tmdb_5000_credits.csv")

In [4]:
movies = movies.merge(credits, on="title")

movies = movies[
    [
        "movie_id",
        "title",
        "overview",
        "genres",
        "keywords",
        "cast",
        "crew"
    ]
]

movies.dropna(inplace=True)

In [5]:
def convert(text):

    L = []

    for i in ast.literal_eval(text):
        L.append(i["name"])

    return L


def convert_cast(text):

    L = []

    counter = 0

    for i in ast.literal_eval(text):

        if counter != 3:
            L.append(i["name"])
            counter += 1

        else:
            break

    return L


def fetch_director(text):

    L = []

    for i in ast.literal_eval(text):

        if i["job"] == "Director":
            L.append(i["name"])
            break

    return L

In [6]:
movies["genres"] = movies["genres"].apply(convert)

movies["keywords"] = movies["keywords"].apply(convert)

movies["cast"] = movies["cast"].apply(convert_cast)

movies["crew"] = movies["crew"].apply(fetch_director)

movies["overview"] = movies["overview"].apply(lambda x: x.split())

In [7]:
for column in ["genres", "keywords", "cast", "crew"]:

    movies[column] = movies[column].apply(
        lambda x: [i.replace(" ", "") for i in x]
    )

In [8]:
movies["tags"] = (
    movies["overview"]
    + movies["genres"]
    + movies["keywords"]
    + movies["cast"]
    + movies["crew"]
)

In [9]:
new_df = movies[
    [
        "movie_id",
        "title",
        "tags"
    ]
]

In [10]:
new_df["tags"] = new_df["tags"].apply(lambda x: " ".join(x))

new_df["tags"] = new_df["tags"].apply(lambda x: x.lower())

## Count Vectorizer

In [11]:
cv = CountVectorizer(
    max_features=5000,
    stop_words="english"
)

In [12]:
vectors = cv.fit_transform(
    new_df["tags"]
).toarray()

In [13]:
vectors.shape

(4806, 5000)

## Cosine Similarity

In [14]:
similarity = cosine_similarity(vectors)

In [15]:
similarity.shape

(4806, 4806)

## Recommendation Function

In [16]:
def recommend(movie):

    movie_index = new_df[
        new_df["title"] == movie
    ].index[0]

    distances = similarity[movie_index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    for movie in movie_list:

        print(new_df.iloc[movie[0]].title)

In [17]:
recommend("Avatar")

Titan A.E.
Small Soldiers
Independence Day
Ender's Game
Aliens vs Predator: Requiem


## Save Model

In [18]:
pickle.dump(
    new_df,
    open("../models/movie_list.pkl", "wb")
)

pickle.dump(
    similarity,
    open("../models/similarity.pkl", "wb")
)

# Model Building Summary

Completed:

- Converted movie tags into vectors using CountVectorizer
- Calculated cosine similarity
- Built a recommendation function
- Saved the movie list and similarity matrix